# 01b - Exploration de la bronze

Avant d'écrire des règles de nettoyage, on **regarde** les données : quelles valeurs prend chaque colonne utile, combien sont vides, comment sont faites les mutations. Les règles de la silver s'appuient sur ces constats.


In [ ]:
import sys
sys.path.append("/home/jovyan/work")

from pyspark.sql import functions as F

from outils.source import creer_session, LAKE

ANNEE = 2025
BRONZE = f"{LAKE}/bronze/dvf_{ANNEE}"

spark = creer_session("exploration")
df = spark.read.format("delta").load(BRONZE)
print(len(df.columns), "colonnes")

NOMS DES COLONNES 

In [ ]:
df.printSchema()

## 1. Les valeurs de `nature_mutation`
Quelles natures de mutation existent, et combien de lignes / de mutations pour chacune ?

In [ ]:
(df.groupBy("nature_mutation")
   .agg(F.count("*").alias("lignes"), F.countDistinct("id_mutation").alias("mutations"))
   .orderBy(F.desc("mutations"))
   .show(truncate=False))

## 2. Les valeurs de `type_local` 


In [ ]:
(df.groupBy("type_local")
   .agg(F.count("*").alias("lignes"), F.countDistinct("id_mutation").alias("mutations"))
   .orderBy(F.desc("lignes"))
   .show(truncate=False))
#vide = terrain sans local

## 3. Colonnes utiles : part de valeurs vides
Les colonnes utiles viennent de la question de Claire
 Pour chacune : quel pourcentage de lignes est vide ?

In [ ]:
UTILES = ["id_mutation", "date_mutation", "nature_mutation", "valeur_fonciere", "code_departement",
          "code_commune", "nom_commune", "id_parcelle", "type_local", "surface_reelle_bati",
          "nombre_pieces_principales", "surface_terrain", "longitude", "latitude"]

vides = df.agg(*[F.round(100 * F.avg(F.col(c).isNull().cast("int")), 1).alias(c) for c in UTILES]).first().asDict()
for col, pct in vides.items():
    print(f"{col:<28} {pct:>5} % vide")

bornes = df.agg(F.min("date_mutation"), F.max("date_mutation")).first()
print("Période couverte :", bornes[0], "->", bornes[1])

## 4. Le prix : vide, nul, négatif ?

In [ ]:
prix = df.withColumn("prix", F.expr("try_cast(valeur_fonciere AS double)"))
print("Prix vide ou non convertible :", prix.filter(F.col("prix").isNull()).count(), "lignes")
print("Prix égal à 0               :", prix.filter(F.col("prix") == 0).count(), "lignes")
print("Prix négatif                :", prix.filter(F.col("prix") < 0).count(), "lignes")

## 5. Structure des mutations
Combien de lignes par mutation ? Combien de lignes de logement (Maison ou Appartement) par mutation ?

In [ ]:
lignes_par_mutation = df.groupBy("id_mutation").count()
lignes_par_mutation.groupBy(F.col("count").alias("lignes_par_mutation")).count().orderBy("lignes_par_mutation").show(10)

nb_log = (df.groupBy("id_mutation")
            .agg(F.sum(F.when(F.col("type_local").isin("Maison", "Appartement"), 1).otherwise(0)).alias("nb_logements")))
nb_log.groupBy("nb_logements").count().orderBy("nb_logements").show(10)

verifcation s'il y a lieu d'ecarter les ventes en l'etat futur d'achevement , et les ventes de terrain lors du nettoyage non bati car claire notre personna ,s'interesse a des ventes de logement 

In [ ]:
vefa = df.filter(F.col("nature_mutation") == "Vente en l'état futur d'achèvement")
vefa.groupBy("type_local").count().show()

In [ ]:
spark.stop()